In [1]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [9]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

inhert_features = ['flag_own_car', 'amt_goods_price_is_missing', 'have_sentinel_value_days_employed', 'own_car_age_is_missing', 'flag_emp_phone', 'flag_cont_mobile', 'ext_source_1_is_missing', 'ext_source_2_is_missing', 'ext_source_3_is_missing', 'info_of_social_circule_is_missing', 'flag_document_9', 'flag_document_13', 'flag_document_16', 'client_without_querys', 'organization_type_Advertising', 'organization_type_Electricity', 'organization_type_Emergency', 'organization_type_Housing', 'organization_type_Industry: type 1', 'organization_type_Industry: type 7', 'organization_type_Other industry', 'organization_type_Other trade', 'organization_type_Postal', 'organization_type_Security', 'organization_type_Services', 'organization_type_Trade: type 2', 'organization_type_Trade: type 6', 'organization_type_Transport: type 2', 'organization_type_XNA', 'bureau_credit_currency_loan_1']
increase_auc = ['days_registration', 'housetype_mode','name_contract_type']
X= X.drop(columns=[*inhert_features,*increase_auc])


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"clean_baseline_no_annuity")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

🏃 View run clean_baseline_no_annuity_child_1 at: http://localhost:5332/#/experiments/3/runs/5910365209924354873615238d1b8af3
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run clean_baseline_no_annuity_child_2 at: http://localhost:5332/#/experiments/3/runs/98d4f3384ac94727a9bacfffbc7a9c63
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run clean_baseline_no_annuity_child_3 at: http://localhost:5332/#/experiments/3/runs/a53951277aa44bf4aed148f9f3712724
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run clean_baseline_no_annuity_child_4 at: http://localhost:5332/#/experiments/3/runs/9150ea0435504c1ea550e894f9b2a0a5
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run clean_baseline_no_annuity_child_5 at: http://localhost:5332/#/experiments/3/runs/8b9d45f67a4f4b1c9e936a0d0ec01e83
🧪 View experiment at: http://localhost:5332/#/experiments/3
AUC per fold= 0.775 ± 0.003(std), auc_score_OOF= 0.775 result of CV with 5 

14290